# Brian2 point-neuron simulation of the FlyWire connectome

This notebook builds a `Brian2CircuitSimulationScanConfig` that uses **every block the Brian2
backend supports**, generates the SONATA simulation from it, and then runs that simulation with
`simulate_brian2.py`.

The circuit is `FlyWire-v783-Brian2-LIF` on staging: a whole-brain leaky integrate-and-fire model
of the adult *Drosophila melanogaster* (138,639 neurons, 15,091,983 synapses) from Shiu, Sterne et
al. (Nature, 2024), built on v783 of the FlyWire connectome. It is staged from the database by the
generation task.

What gets exercised:

| Group | Blocks |
| --- | --- |
| Neuron sets | `PointPopulationPredefinedNeuronSet`, `PointPopulationIDNeuronSet` |
| Timestamps | `RegularTimestamps` |
| Distributions | `ExponentialDistribution` |
| Stimuli | `Brian2DirectPoissonStimulus`, `ConstantCurrentClampSomaticStimulus`, `LinearCurrentClampSomaticStimulus`, `MultiPulseCurrentClampSomaticStimulus`, `SimulationDtSinusoidalCurrentClampSomaticStimulus`, `PoissonSpikeStimulus`, `InterSpikeIntervalDistributionSpikeStimulus` |
| Recordings | `SimulationDtSomaVoltageRecording`, `SimulationDtTimeWindowSomaVoltageRecording` |
| Synaptic manipulations | `DisconnectSynapticManipulation` |

Staging the circuit downloads ~140 MB the first time. Generation takes a few seconds; the 100 ms
simulation takes roughly a minute, most of it building the 15M synapses.

## 1. Connect to the database

In [ ]:
from pathlib import Path

import obi_one as obi
from entitysdk import Client, ProjectContext
from obi_auth import get_token

token = get_token(environment="staging")
project_context = ProjectContext(
    virtual_lab_id=obi.LAB_ID_STAGING_TEST,
    project_id=obi.PROJECT_ID_STAGING_TEST,
)
db_client = Client(
    api_url="https://staging.openbraininstitute.org/api/entitycore",
    project_context=project_context,
    token_manager=token,
)

## 2. The circuit

The Brian2 backend requires a circuit with a single point-neuron node population and a single
edge population, and no virtual populations. `FlyWire-v783-Brian2-LIF` has one `brian2_point`
population named `drosophila`.

In [ ]:
from entitysdk.models import Circuit

circuit_id = "e3cfde75-284c-434e-91c5-dd0e606dbf43"
POINT_POPULATION = "drosophila"

circuit_entity = db_client.get_entity(entity_id=circuit_id, entity_type=Circuit)
print(f"{circuit_entity.name}: {circuit_entity.number_neurons:,} neurons, "
      f"{circuit_entity.number_synapses:,} synapses")

## 3. Neuron sets

Only point-neuron sets are available. `sugar` and `gustatory` are node sets that ship with the
circuit; `Probe` is an explicit list of node IDs we record from so the voltage traces stay small.

`sugar` — the 20 gustatory receptor neurons stimulated in the original model — is also the set an
untargeted `Brian2DirectPoissonStimulus` falls back to.

In [ ]:
SIM_DURATION = 100.0  # ms

sim_conf = obi.Brian2CircuitSimulationScanConfig.empty_config()
sim_conf.set(
    obi.Info(
        campaign_name="FlyWire Brian2 feature tour",
        campaign_description="Every Brian2-supported block on the FlyWire LIF circuit",
    ),
    name="info",
)

sugar = obi.PointPopulationPredefinedNeuronSet(node_set="sugar", population=POINT_POPULATION)
sim_conf.add(sugar, name="Sugar")

gustatory = obi.PointPopulationPredefinedNeuronSet(
    node_set="gustatory", population=POINT_POPULATION
)
sim_conf.add(gustatory, name="Gustatory")

probe = obi.PointPopulationIDNeuronSet(
    population=POINT_POPULATION,
    neuron_ids=obi.NamedTuple(name="probe", elements=range(20)),
)
sim_conf.add(probe, name="Probe")

## 4. Timestamps and distributions

Timestamps say *when* a stimulus or manipulation fires; a block referencing them is emitted once
per timestamp. Distributions are only needed by the two distribution-driven spike stimuli.

In [ ]:
ticks = obi.RegularTimestamps(start_time=10.0, number_of_repetitions=2, interval=40.0)
sim_conf.add(ticks, name="Ticks")

isi = obi.ExponentialDistribution(scale=25.0)
sim_conf.add(isi, name="ISI")

## 5. Stimuli

The Brian2 backend understands five SONATA input modules, and the blocks below cover all of them:
`poisson`, `linear`, `pulse`, `sinusoidal` and `synapse_replay`.

Current amplitudes are in nanoamps. With this model's 10 MΩ membrane resistance, 1 nA
depolarises by ~10 mV — comfortably across the 7 mV gap from rest (−52 mV) to threshold (−45 mV).

One thing to know before wiring up the replays:

> **A spike stimulus's target reads as a filter on its source.** The runner materialises the
> input's `node_set` against the *spike file's* population and uses it to mask which spikes are
> replayed, but the generation task writes the stimulus's **target** neuron set there. Leave the
> target unset — it then covers every point neuron and filters nothing — or make sure it contains
> the source. A target that excludes the source replays nothing at all, silently.

In [ ]:
# `poisson`: an independent Poisson train kicking each target neuron's membrane potential
# directly, bypassing the circuit's synapses. This is what drives activity in the original model.
sim_conf.add(
    obi.Brian2DirectPoissonStimulus(neuron_set=sugar.ref, frequency=150.0, weight=68.75),
    name="SugarPoisson",
)

# `linear`: a constant step, and a ramp between two amplitudes.
sim_conf.add(
    obi.ConstantCurrentClampSomaticStimulus(
        neuron_set=probe.ref, timestamps=ticks.ref, amplitude=1.0, duration=10.0
    ),
    name="Step",
)
sim_conf.add(
    obi.LinearCurrentClampSomaticStimulus(
        neuron_set=probe.ref,
        timestamps=ticks.ref,
        amplitude_start=0.0,
        amplitude_end=2.0,
        duration=10.0,
    ),
    name="Ramp",
)

# `pulse`: a 100 Hz train of 2 ms pulses.
sim_conf.add(
    obi.MultiPulseCurrentClampSomaticStimulus(
        neuron_set=probe.ref,
        timestamps=ticks.ref,
        amplitude=2.0,
        width=2.0,
        frequency=100.0,
        duration=10.0,
    ),
    name="Pulses",
)

# `sinusoidal`: sampled at the simulation timestep, so this block has no Timestep of its own.
sim_conf.add(
    obi.SimulationDtSinusoidalCurrentClampSomaticStimulus(
        neuron_set=probe.ref,
        timestamps=ticks.ref,
        maximum_amplitude=2.0,
        frequency=50.0,
        duration=10.0,
    ),
    name="Sine",
)

# `synapse_replay`: spikes generated for the source neurons are replayed through the circuit's
# own connectivity. Targets are left unset for the reason described above.
sim_conf.add(
    obi.PoissonSpikeStimulus(
        source_neuron_set=sugar.ref, timestamps=ticks.ref, frequency=100.0, duration=30.0
    ),
    name="Replay",
)
sim_conf.add(
    obi.InterSpikeIntervalDistributionSpikeStimulus(
        source_neuron_set=gustatory.ref,
        timestamps=ticks.ref,
        distribution=isi.ref,
        duration=30.0,
    ),
    name="ISIReplay",
)

## 6. Recordings

Brian2 samples its `StateMonitor` on the integration timestep and rejects a report asking for any
other interval, so these blocks have no Timestep parameter — that is what the `SimulationDt`
prefix means. Only soma voltage is reported.

In [ ]:
sim_conf.add(obi.SimulationDtSomaVoltageRecording(neuron_set=probe.ref), name="ProbeVoltage")
sim_conf.add(
    obi.SimulationDtTimeWindowSomaVoltageRecording(
        neuron_set=sugar.ref, start_time=0.0, end_time=50.0
    ),
    name="SugarVoltageEarly",
)

## 7. Synaptic manipulations

`Connect` and `Disconnect` become SONATA `connection_overrides`, applied part-way through the run
at the timestamps they reference. Brian2 honours an override's `weight` and
`synapse_delay_override`; the mechanism-specific manipulations are not offered because it rejects
`synapse_configure` and `modoverride` outright.

In [ ]:
sim_conf.add(
    obi.DisconnectSynapticManipulation(
        presynaptic_neuron_set=sugar.ref,
        postsynaptic_neuron_set=gustatory.ref,
        timestamps=ticks.ref,
    ),
    name="CutSugarToGustatory",
)

## 8. Initialize

`v_init` is set to the model's resting potential (−52 mV) rather than the −80 mV default, which
belongs to biophysical models.

The circuit is referenced by ID, so the generation task stages it from the database.

In [ ]:
sim_conf.set(
    obi.Brian2CircuitSimulationScanConfig.Initialize(
        circuit=obi.CircuitFromID(id_str=circuit_id),
        simulation_length=SIM_DURATION,
        v_init=-52.0,
    ),
    name="initialize",
)

validated_sim_conf = sim_conf.validated_config()

## 9. Generate the simulation

This registers a `SimulationCampaign` and a `Simulation` in the database, stages the circuit
(~140 MB on first run), and writes `simulation_config.json`, `node_sets.json` and the spike files
for the two replay stimuli.

In [ ]:
output_root = Path("../../../../../../../obi-output/brian2_simulations/flywire_feature_tour")

grid_scan = obi.GridScanGenerationTask(
    form=validated_sim_conf,
    output_root=str(output_root),
    coordinate_directory_option="ZERO_INDEX",
)
grid_scan.execute(db_client=db_client)
obi.run_tasks_for_generated_scan(grid_scan, db_client=db_client)

coordinate_dir = output_root / "0"
simulation_config_path = coordinate_dir / "simulation_config.json"
print(simulation_config_path.resolve())

## 10. What was generated

In [ ]:
import json

sonata_config = json.loads(simulation_config_path.read_text())

print("run:", sonata_config["run"])
print()
print("inputs:")
for name, entry in sonata_config["inputs"].items():
    print(f"  {name:20} {entry['module']:16} -> {entry.get('node_set', entry.get('compartment_set'))}")
print()
print("reports:")
for name, entry in sonata_config["reports"].items():
    print(f"  {name:20} dt={entry['dt']} window=({entry['start_time']}, {entry['end_time']})")
print()
print("connection_overrides:")
for entry in sonata_config["connection_overrides"]:
    print(f"  {entry['name']:24} {entry['source']} -> {entry['target']} "
          f"weight={entry['weight']} at t={entry['delay']} ms")

Every report's `dt` equals `run.dt`, and so does the sinusoidal input's — that is the constraint
the `SimulationDt` blocks exist to satisfy. The stimuli referencing `Ticks` appear twice each
(`_0` and `_1`), once per timestamp.

## 11. Run the simulation

`simulate_brian2.py` is the script that turns the generated SONATA config into a Brian2 network
and runs it. Its `sonata-simulation` command takes the config path and writes the spike report and
the voltage reports into the simulation's output directory.

Most of the runtime is spent instantiating all 15M synapses.

In [ ]:
import subprocess
import sys

simulate_brian2 = (
    Path(obi.__file__).parent
    / "scientific" / "library" / "simulation" / "brian2" / "simulate_brian2.py"
)

result = subprocess.run(
    [sys.executable, str(simulate_brian2), "sonata-simulation",
     "--simulation-path", str(simulation_config_path.resolve()), "-v"],
    capture_output=True,
    text=True,
    check=False,
)
print(result.stderr[-2000:] if result.returncode else result.stderr[-800:])
result.check_returncode()

## 12. Results

The spike report and the two voltage reports are read back with bluepysnap.

In [ ]:
import bluepysnap

simulation = bluepysnap.Simulation(simulation_config_path)

spikes = simulation.spikes[POINT_POPULATION].get()
print(f"{len(spikes):,} spikes from {spikes.nunique():,} neurons")

probe_voltage = simulation.reports["ProbeVoltage"].filter().report
sugar_voltage = simulation.reports["SugarVoltageEarly"].filter().report
print("ProbeVoltage report:", probe_voltage.shape)
print("SugarVoltageEarly report:", sugar_voltage.shape)

In [ ]:
import matplotlib.pyplot as plt

fig, (raster, trace) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

raster.scatter(spikes.index, spikes.to_numpy(), s=1, alpha=0.4, color="#1f77b4")
raster.set_ylabel("node id")
raster.set_title(f"FlyWire LIF network, {SIM_DURATION:.0f} ms ({len(spikes):,} spikes)")

for node in probe_voltage.columns[:5]:
    trace.plot(probe_voltage.index, probe_voltage[node], lw=0.8, label=f"neuron {node[1]}")
trace.axhline(-45.0, color="grey", ls="--", lw=0.8, label="threshold")
trace.set_xlabel("time (ms)")
trace.set_ylabel("V (mV)")
trace.set_title("Soma voltage of the current-injected probe neurons")
trace.legend(loc="upper right", fontsize="small", ncol=3)

fig.tight_layout()

The probe traces show the four current injections arriving together at each of the two timestamps
(10 ms and 50 ms), on top of the network activity the Poisson drive and the two spike replays
produce through the `sugar` and `gustatory` neurons.